### Implementation of Decision Trees in scikit-learn

Phys 4730 Lab 22
2026/11/12

Instructor Solutions

Ref: V. Acquaviva, <i>Machine Learning for Physics and Astronomy</i> 

In [ ]:
%reset -f

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams.update({'font.size': 14})

In [ ]:
### some new imports for this lab

import pandas as pd                             # python data analysis library
from sklearn.tree import DecisionTreeClassifier # needed scikit-learn packages
from sklearn import metrics                     # needed scikit-learn packages
from sklearn.tree import export_graphviz        # needed scikit-learn packages
from sklearn import tree                        # needed scikit-learn packages

In [ ]:
### use pandas to import data from .csv file ###

LearningSet = pd.read_csv('HPLearningSet.csv')
LearningSet = LearningSet.drop(LearningSet.columns[0], axis=1) # drop annoying (redundant) first column
LearningSet

In [ ]:
### Quick note on Pandas for the uninitiated:    ###
### With a Pandas "data frame", we can refer     ###
### to columns by either their names or indices. ###

LearningSet[['P_NAME','S_MASS']]

In [ ]:
### ...and now the other way... ###

LearningSet.iloc[:,[0,1]]

In [ ]:
### Moving on. Pick the training set and the test set.                      ###
### (For starters, first 13 elements are for training, last 5 for testing.) ###

TrainSet = LearningSet.iloc[:13,:]  
TestSet  = LearningSet.iloc[13:,:]

In [ ]:
TrainSet

In [ ]:
TestSet

In [ ]:
### split the training and testing sets into features and labels (output) ###
### for now, we will only deal with the stellar mass and orbital period   ###

Xtrain = TrainSet.drop(['P_NAME','P_DISTANCE','P_HABITABLE'],axis=1)  # features for the train set
Xtest  = TestSet.drop (['P_NAME','P_DISTANCE','P_HABITABLE'],axis=1)  # features for the test set

ytrain = TrainSet.P_HABITABLE                           # label/target for the train set
ytest  = TestSet.P_HABITABLE                            # label/target for the test set

In [ ]:
Xtrain

In [ ]:
ytrain

In [ ]:
### plot the data ###

plt.figure(figsize=(8,6))
plt.scatter(LearningSet.S_MASS,LearningSet.P_PERIOD)
plt.xlabel("Mass of Parent Star $M_{\odot}$")
plt.ylabel("Period of Orbit (days)")
plt.yscale("log")
plt.axis([0.0,1.4,1.0,2000.0])
plt.grid()

plt.savefig("data.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### split training set exoplanets into inhabitable/uninhabitable ###

Xtrain_inhabitable   = Xtrain[TrainSet.P_HABITABLE != 0]
Xtrain_uninhabitable = Xtrain[TrainSet.P_HABITABLE == 0]

### while we're at it, do the test set ###

Xtest_inhabitable    = Xtest[TestSet.P_HABITABLE != 0]
Xtest_uninhabitable  = Xtest[TestSet.P_HABITABLE == 0]

In [ ]:
### plot the training set ###

plt.figure(figsize=(8,6))

xpmin = 0.0
xpmax = 1.4
ypmin = 1.0
ypmax = 2000.0

plt.scatter(Xtrain_inhabitable['S_MASS'],  Xtrain_inhabitable['P_PERIOD'],   marker = 'o', c = 'b', s = 40, label ="inhabitable")
plt.scatter(Xtrain_uninhabitable['S_MASS'],Xtrain_uninhabitable['P_PERIOD'], marker = 'x', c = 'r', s = 40, label ="uninhabitable")

plt.title("Training Set")
plt.xlabel("Mass of Parent Star $M_{\odot}$")
plt.ylabel("Period of Orbit (days)")
plt.yscale("log")
plt.axis([xpmin,xpmax,ypmin,ypmax])
plt.grid()
plt.legend(loc="upper left")

plt.savefig("training.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### now fit the data with decision tree ###

model = DecisionTreeClassifier(random_state = 3)  # select method, fix random state for repeatability
model.fit(Xtrain, ytrain)                         # build the model 

In [ ]:
### obtain information from the tree structure ###

tree1 = model.tree_
split_values = tree1.threshold
split_values

In [ ]:
### sklearn enables us to visualize the decision tree ###

plt.figure(figsize=(10,8))  # customize according to the size of your tree

tree.plot_tree(model, feature_names = ['Mass', 'Period', 'Distance'], class_names = ['Not Habitable','Habitable'])

plt.savefig("tree.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### reproduce the training set graph, with the splits ###

plt.figure(figsize=(8,6))

xpmin = 0.0
xpmax = 1.4
ypmin = 1.0
ypmax = 2000.0

plt.scatter(Xtrain_inhabitable['S_MASS'],  Xtrain_inhabitable['P_PERIOD'],   marker = 'o', c = 'b', s = 40, label ="inhabitable")
plt.scatter(Xtrain_uninhabitable['S_MASS'],Xtrain_uninhabitable['P_PERIOD'], marker = 'x', c = 'r', s = 40, label ="uninhabitable")

plt.title("Training Set")
plt.xlabel("Mass of Parent Star $M_{\odot}$")
plt.ylabel("Period of Orbit (days)")
plt.yscale("log")
plt.axis([xpmin,xpmax,ypmin,ypmax])
plt.grid()

# the splits 

plt.plot([0.83,0.83],[ypmin,ypmax], linewidth =1, ls = '--', c = 'k')
plt.plot([0.00,0.83],[4.891,4.891], linewidth =1, ls = '--', c = 'k')

plt.legend(loc="upper left")
plt.savefig("training_splits.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### same graph for the test data ###

plt.figure(figsize=(8,6))

xpmin = 0.0
xpmax = 1.4
ypmin = 1.0
ypmax = 2000.0

plt.scatter(Xtest_inhabitable['S_MASS'],  Xtest_inhabitable['P_PERIOD'],   marker = 'o', c = 'b', s = 40, label ="inhabitable")
plt.scatter(Xtest_uninhabitable['S_MASS'],Xtest_uninhabitable['P_PERIOD'], marker = 'x', c = 'r', s = 40, label ="uninhabitable")

plt.title("Test Set")
plt.xlabel("Mass of Parent Star $M_{\odot}$")
plt.ylabel("Period of Orbit (days)")
plt.yscale("log")
plt.axis([xpmin,xpmax,ypmin,ypmax])
plt.grid()

# the splits 

plt.plot([0.83,0.83],[ypmin,ypmax], linewidth =1, ls = '--', c = 'k')
plt.plot([0.00,0.83],[4.891,4.891], linewidth =1, ls = '--', c = 'k')

plt.legend(loc="upper left")
plt.savefig("test_splits.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### We can see by eye that while the splits were 100% accurate    ###
### on the training set, they were only 60% accurate on the test. ###
### What does sklearn have to say about it?                       ###

train_acc = metrics.accuracy_score(ytrain, model.predict(Xtrain))
test_acc  = metrics.accuracy_score(ytest,  model.predict(Xtest))
print("training accuracy: %4.2f, test accuracy: %4.2f" % (train_acc,test_acc))


### Lab 22 Exercise 1: repeat the above for a different train/test division

<br>

In [ ]:
### pick the training set and the test set ###

TrainSet1 = LearningSet.iloc[5:,:] 
TestSet1  = LearningSet.iloc[:5,:]

In [ ]:
### split the training and testing sets into features and labels        ###
### for now, we will only deal with the stellar mass and orbital period ###

Xtrain1 = TrainSet1.drop(['P_NAME','P_DISTANCE','P_HABITABLE'],axis=1) # features for the train set
Xtest1  = TestSet1.drop (['P_NAME','P_DISTANCE','P_HABITABLE'],axis=1)  # features for the test set

ytrain1 = TrainSet1.P_HABITABLE                           # label/target for the train set                         
ytest1  = TestSet1.P_HABITABLE                            # label/target for the test set   

In [ ]:
### now fit the data with decision tree ###

model1 = DecisionTreeClassifier(random_state = 3)  # select method, fix random state for repeatability
model1.fit(Xtrain1, ytrain1)                       # build the model 

In [ ]:
### obtain information from the tree structure ###

tree1 = model1.tree_
split_values1 = tree1.threshold
split_values1

In [ ]:
### visualize the new decision tree ###

plt.figure(figsize=(40,20))  # customize according to the size of your tree

tree.plot_tree(model1, feature_names = ['Mass', 'Period'], class_names = ['Not Habitable','Habitable'])

plt.savefig("tree1.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### split training set exoplanets into inhabitable/uninhabitable ###

Xtrain1_inhabitable   = Xtrain1[TrainSet1.P_HABITABLE != 0]
Xtrain1_uninhabitable = Xtrain1[TrainSet1.P_HABITABLE == 0]

### while we're at it, do the test set ###

Xtest1_inhabitable    = Xtest1[TestSet1.P_HABITABLE != 0]
Xtest1_uninhabitable  = Xtest1[TestSet1.P_HABITABLE == 0]

In [ ]:
### reproduce the training set graph, with the splits ###

plt.figure(figsize=(8,6))

xpmin = 0.0
xpmax = 1.4
ypmin = 1.0
ypmax = 2000.0

plt.scatter(Xtrain1_inhabitable['S_MASS'],  Xtrain1_inhabitable['P_PERIOD'],   marker = 'o', c = 'b', s = 40, label ="inhabitable")
plt.scatter(Xtrain1_uninhabitable['S_MASS'],Xtrain1_uninhabitable['P_PERIOD'], marker = 'x', c = 'r', s = 40, label ="uninhabitable")

plt.title("Exercise 1 Training Set")
plt.xlabel("Mass of Parent Star $M_{\odot}$")
plt.ylabel("Period of Orbit (days)")
plt.yscale("log")
plt.axis([xpmin,xpmax,ypmin,ypmax])
plt.grid()

# the splits 

plt.plot([0.575,0.575],[  ypmin,  ypmax], linewidth =1, ls = '--', c='k')
plt.plot([0.000,0.575],[  4.891,  4.891], linewidth =1, ls = '--', c='k')
plt.plot([0.000,0.575],[ 23.747, 23.747], linewidth =1, ls = '--', c='k')
plt.plot([0.265,0.265],[  4.891, 23.747], linewidth =1, ls = '--', c='k')
plt.plot([0.575,xpmax],[112.309,112.309], linewidth =1, ls = '--', c='k')
plt.plot([0.965,0.965],[112.309,  ypmax], linewidth =1, ls = '--', c='k')

plt.legend(loc="upper left")
plt.savefig("training1_splits.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### same graph for the test data ###

plt.figure(figsize=(8,6))

xpmin = 0.0
xpmax = 1.4
ypmin = 1.0
ypmax = 2000.0

plt.scatter(Xtest1_inhabitable['S_MASS'],  Xtest1_inhabitable['P_PERIOD'],   marker = 'o', c = 'b', s = 40, label ="inhabitable")
plt.scatter(Xtest1_uninhabitable['S_MASS'],Xtest1_uninhabitable['P_PERIOD'], marker = 'x', c = 'r', s = 40, label ="uninhabitable")

plt.title("Exercise 1 Test Set")
plt.xlabel("Mass of Parent Star $M_{\odot}$")
plt.ylabel("Period of Orbit (days)")
plt.yscale("log")
plt.axis([xpmin,xpmax,ypmin,ypmax])
plt.grid()

# the splits 

plt.plot([0.575,0.575],[  ypmin,  ypmax], linewidth =1, ls = '--', c='k')
plt.plot([0.000,0.575],[  4.891,  4.891], linewidth =1, ls = '--', c='k')
plt.plot([0.000,0.575],[ 23.747, 23.747], linewidth =1, ls = '--', c='k')
plt.plot([0.265,0.265],[  4.891, 23.747], linewidth =1, ls = '--', c='k')
plt.plot([0.575,xpmax],[112.309,112.309], linewidth =1, ls = '--', c='k')
plt.plot([0.965,0.965],[112.309,  ypmax], linewidth =1, ls = '--', c='k')

plt.legend(loc="upper left")
plt.savefig("test1_splits.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### output accuracy ###

train_acc = metrics.accuracy_score(ytrain1, model1.predict(Xtrain1))
test_acc  = metrics.accuracy_score(ytest1,  model1.predict(Xtest1))
print("training accuracy: %4.2f, test accuracy: %4.2f" % (train_acc,test_acc))


### Lab 22 Exercise 2: repeat the above for Ex 1 train/test division, add Distance to the features

<br>

In [ ]:
### pick the training set and the test set ###

TrainSet2 = LearningSet.iloc[5:,:] 
TestSet2  = LearningSet.iloc[:5,:]

In [ ]:
### split the training and testing sets into features and labels  ###
### now add distance to the star                                  ###

Xtrain2 = TrainSet2.drop(['P_NAME','P_HABITABLE'],axis=1) # features for the train set
Xtest2  = TestSet2.drop (['P_NAME','P_HABITABLE'],axis=1) # features for the test set

ytrain2 = TrainSet2.P_HABITABLE                           # label/target for the train set                         
ytest2  = TestSet2.P_HABITABLE                            # label/target for the test set   

In [ ]:
### now fit the data with decision tree ###

model2 = DecisionTreeClassifier(random_state = 3)  # select method, fix random state for repeatability
model2.fit(Xtrain2, ytrain2)                       # build the model 

In [ ]:
### obtain information from the tree structure ###

tree2 = model2.tree_
split_values2 = tree2.threshold
split_values2

In [ ]:
### visualize the new decision tree ###

plt.figure(figsize=(40,20))  # customize according to the size of your tree

tree.plot_tree(model2, feature_names = ['Mass', 'Period', 'Distance'], class_names = ['Not Habitable','Habitable'])

plt.savefig("tree2.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### output accuracy ###

train_acc = metrics.accuracy_score(ytrain2, model2.predict(Xtrain2))
test_acc  = metrics.accuracy_score(ytest2,  model2.predict(Xtest2))
print("training accuracy: %4.2f, test accuracy: %4.2f" % (train_acc,test_acc))

### Lab 22 Exercise 3: answer some review questions

#### Review Questions

1) The DT algorithm has as an advantage the fact that it is easy to understand and interpret. Can you think of any disadvantage(s)?

<font color='red'>It is not well suited for situations in which &ldquo;diagonal&rdquo; cuts in the features might be desireable. The fact that it always has 100% training set accuracy indicates that it is subject to overfitting. </font>

2) Comment on the suitability of the data we have chosen for this lab. How might the dataset be improved?

<font color='red'> Biggest thing is: We need more of it! </font>

3) Would Earth be classified as a habitable planet?

<font color='red'> No! Look at the final cuts for Exercise 1... mass = 1 solar mass and period = 365 days is in a region designated &ldquo;uninhabitable&rdquo;. </font>  